
# Flight Delay ML Platform — Feature Engineering

## 1. Feature Availability Contract

### Prediction Objective

Predict whether a domestic flight will experience a departure or arrival delay of at least 15 minutes.

The intended prediction timestamp is 24 hours before scheduled departure (T-24h).

Two separate classification tasks are defined:

- Departure delay: `DepDel15`
- Arrival delay: `ArrDel15`

### Modeling Population

The dataset includes flights that:

- Depart from ATL, DFW, DEN, ORD, LAX, or SFO
- Were not cancelled or diverted
- Have both delay labels available

Predictions therefore represent delay risk conditional on flight completion.

### Feature Categories

**Calendar Features**
- Month
- Day of week
- Day of month
- Weekend indicator

**Historical Scheduled-Flight Features**
- Reporting airline
- Origin airport
- Destination airport
- Scheduled departure time
- Scheduled arrival time
- Scheduled elapsed time
- Distance

**Historical Operational Features (Future Work)**
- Prior airline delay rates
- Prior airport delay rates
- Prior route delay rates

Historical aggregates require point-in-time availability checks and must exclude outcomes not available at prediction time.

### Excluded Features

Actual flight outcomes must never be included in the predictor matrix.

Examples:
- DepDelay
- ArrDelay
- DepTime
- ArrTime
- TaxiOut
- ActualElapsedTime
- DepDel15
- ArrDel15

Flight numbers are retained for audit and record tracing but excluded from the initial feature set.

### Data Availability Limitation

The BTS dataset contains retrospectively recorded scheduled flight information rather than verified T-24h schedule snapshots.

Consequently, the initial scheduled-feature model is a retrospective baseline, not a fully validated real-time T-24h deployment model.

### Data Split Policy

- Train: 2023
- Validation: January–August 2024
- Calibration: September–October 2024
- Threshold selection: November–December 2024
- Final test: 2025

The final test dataset is reserved for evaluation after model development decisions are finalized.


In [ ]:
from pathlib import Path

import pandas as pd

from flight_delay.data.storage import load_parquet
from flight_delay.features.schema import validate_baseline_features

# Locate project root.
current = Path.cwd().resolve()

ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "pyproject.toml").exists()
    ),
    None,
)

assert ROOT is not None, "Project root not found."

# Load January 2023 training data only.
path = (
    ROOT / "data" / "processed"
    / "bts_2023_01_completed.parquet"
)

df = load_parquet(path)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

In [ ]:
BASELINE_INPUT_COLUMNS = [
    "FlightDate",
    "Reporting_Airline",
    "Origin",
    "Dest",
    "CRSDepTime",
    "CRSArrTime",
    "CRSElapsedTime",
    "Distance",
]

TARGET_COLUMNS = ["DepDel15", "ArrDel15"]

OUTCOME_COLUMNS = {
    "DepDelay",
    "ArrDelay",
    "DepDel15",
    "ArrDel15",
    "Cancelled",
    "Diverted",
}

# Validate the raw feature allowlist.
validate_baseline_features(BASELINE_INPUT_COLUMNS)

X_raw = df[BASELINE_INPUT_COLUMNS].copy()

y_dep = df["DepDel15"].copy()
y_arr = df["ArrDel15"].copy()

# Explicit leakage checks.
assert not set(X_raw.columns) & OUTCOME_COLUMNS
assert set(X_raw.columns) == set(BASELINE_INPUT_COLUMNS)

assert len(X_raw) == len(y_dep) == len(y_arr)

print("=== Feature / Target Separation ===")
print("X shape:", X_raw.shape)
print("Departure target:", y_dep.shape)
print("Arrival target:", y_arr.shape)

print("\nFeature columns:")
print(X_raw.columns.tolist())

print("\nFeature / target separation passed!")


## 2. Calendar and Scheduled Flight Feature Engineering

### Objective

Transform raw scheduled-flight information into structured machine learning features.

### Calendar Features

Derived from `FlightDate`:

- `month`: Month of the year (1–12)
- `day_of_week`: Monday = 0, Sunday = 6
- `day_of_month`: Day of the month (1–31)
- `is_weekend`: Binary weekend indicator

### Cyclical Features

Time-related variables follow periodic patterns.

We apply sine and cosine transformations to:

- Month of year (period = 12)
- Day of week (period = 7)
- Scheduled departure time (period = 1440 minutes)
- Scheduled arrival time (period = 1440 minutes)

For a periodic value `t` with period `P`:

\[
\text{sin_feature} = \sin(2\pi t/P)
\]

\[
\text{cos_feature} = \cos(2\pi t/P)
\]

### Additional Features

- Reporting airline
- Origin airport
- Destination airport
- Scheduled elapsed time
- Flight distance

### Leakage Prevention

Only approved historical scheduled-flight fields are used.

Actual delay outcomes and target labels remain excluded.

The BTS schedule fields are retrospective observations and are not verified T-24h schedule snapshots.


In [ ]:
import numpy as np
import pandas as pd


def hhmm_to_minutes(values: pd.Series) -> pd.Series:
    """Convert HHMM values to minutes after midnight."""

    hhmm = pd.to_numeric(values, errors="raise")

    if hhmm.isna().any():
        raise ValueError("Scheduled times contain missing values.")

    if (hhmm % 1 != 0).any():
        raise ValueError("Scheduled times must be integers.")

    hhmm = hhmm.astype("int32")

    hours = hhmm // 100
    minutes = hhmm % 100

    # Some schedules may represent midnight as 2400.
    valid = (
        ((hours >= 0) & (hours < 24) & (minutes < 60))
        | (hhmm == 2400)
    )

    if not valid.all():
        raise ValueError("Invalid HHMM scheduled time.")

    # Normalize 2400 to 00:00.
    hhmm = hhmm.where(hhmm != 2400, 0)

    return (hhmm // 100) * 60 + (hhmm % 100)


# Test the conversion with known examples.
examples = pd.Series([5, 930, 1435, 2359, 2400])

converted = hhmm_to_minutes(examples)

print(pd.DataFrame({
    "HHMM": examples,
    "MinutesAfterMidnight": converted,
}))

assert converted.tolist() == [5, 570, 875, 1439, 0]

print("\nHHMM conversion passed!")

In [ ]:
# Start from the approved feature allowlist.
feature_df = X_raw[
    [
        "Reporting_Airline",
        "Origin",
        "Dest",
        "CRSElapsedTime",
        "Distance",
    ]
].copy()

# Calendar features.
flight_dates = pd.to_datetime(
    X_raw["FlightDate"],
    errors="raise",
)

feature_df["month"] = flight_dates.dt.month
feature_df["day_of_week"] = flight_dates.dt.dayofweek
feature_df["day_of_month"] = flight_dates.dt.day
feature_df["is_weekend"] = (
    feature_df["day_of_week"] >= 5
).astype("int8")

# Cyclical calendar encoding.
month_angle = (
    2 * np.pi * (feature_df["month"] - 1) / 12
)

weekday_angle = (
    2 * np.pi * feature_df["day_of_week"] / 7
)

feature_df["month_sin"] = np.sin(month_angle)
feature_df["month_cos"] = np.cos(month_angle)

feature_df["weekday_sin"] = np.sin(weekday_angle)
feature_df["weekday_cos"] = np.cos(weekday_angle)

# Scheduled departure and arrival times.
dep_minutes = hhmm_to_minutes(X_raw["CRSDepTime"])
arr_minutes = hhmm_to_minutes(X_raw["CRSArrTime"])

# Cyclical time encoding.
dep_angle = 2 * np.pi * dep_minutes / 1440
arr_angle = 2 * np.pi * arr_minutes / 1440

feature_df["dep_time_sin"] = np.sin(dep_angle)
feature_df["dep_time_cos"] = np.cos(dep_angle)

feature_df["arr_time_sin"] = np.sin(arr_angle)
feature_df["arr_time_cos"] = np.cos(arr_angle)

print("=== Engineered Features ===")
print("Original feature count:", X_raw.shape[1])
print("Engineered feature count:", feature_df.shape[1])
print("Feature matrix shape:", feature_df.shape)

display(feature_df.head())

In [ ]:
FORBIDDEN_FEATURES = {
    "DepDelay",
    "ArrDelay",
    "DepDel15",
    "ArrDel15",
    "Cancelled",
    "Diverted",
    "Flight_Number_Reporting_Airline",
}

assert len(feature_df) == len(X_raw)

assert feature_df.shape[1] == 17

assert not (set(feature_df.columns) & FORBIDDEN_FEATURES)

assert feature_df.isna().sum().sum() == 0

# Validate cyclical feature ranges.
cyclical_cols = [
    col for col in feature_df.columns
    if col.endswith(("_sin", "_cos"))
]

assert feature_df[cyclical_cols].ge(-1.000001).all().all()
assert feature_df[cyclical_cols].le(1.000001).all().all()

print("=== Feature Validation ===")
print("Records:", len(feature_df))
print("Features:", feature_df.shape[1])
print("Missing values:", int(feature_df.isna().sum().sum()))
print("Cyclical features:", len(cyclical_cols))
print("Forbidden features:", set(feature_df.columns) & FORBIDDEN_FEATURES)

print("\nFeature engineering pilot passed!")

In [ ]:
import numpy as np

FORBIDDEN_FEATURES = {
    "DepDelay",
    "ArrDelay",
    "DepDel15",
    "ArrDel15",
    "Cancelled",
    "Diverted",
    "Flight_Number_Reporting_Airline",
}

cyclical_cols = [
    col for col in feature_df.columns
    if col.endswith(("_sin", "_cos"))
]

numeric_cols = feature_df.select_dtypes(
    include="number"
).columns

assert feature_df.shape == (112548, 17)
assert feature_df.isna().sum().sum() == 0
assert not set(feature_df.columns) & FORBIDDEN_FEATURES
assert len(cyclical_cols) == 8

# All numerical values must be finite.
assert np.isfinite(
    feature_df[numeric_cols].to_numpy()
).all()

# Cyclical encodings must be in [-1, 1].
assert feature_df[cyclical_cols].ge(-1.000001).all().all()
assert feature_df[cyclical_cols].le(1.000001).all().all()

print("=== Feature Validation ===")
print("Records:", len(feature_df))
print("Features:", feature_df.shape[1])
print("Missing values:", int(feature_df.isna().sum().sum()))
print("Cyclical features:", len(cyclical_cols))
print(
    "Forbidden features:",
    set(feature_df.columns) & FORBIDDEN_FEATURES,
)

print("\nFeature engineering pilot passed!")

In [ ]:
from pandas.testing import assert_frame_equal

from flight_delay.features.build import (
    FEATURE_COLUMNS,
    build_baseline_features,
)

# Rebuild features using the production module.
engineered_df = build_baseline_features(X_raw)

# Compare against the original notebook implementation.
assert_frame_equal(
    feature_df.loc[:, list(FEATURE_COLUMNS)],
    engineered_df,
    check_dtype=False,
    check_exact=False,
    rtol=1e-12,
    atol=1e-12,
)

print("=== Reusable Feature Module ===")
print("Feature shape:", engineered_df.shape)
print("Feature count:", len(FEATURE_COLUMNS))
print("Missing values:", int(engineered_df.isna().sum().sum()))

print("\nNotebook and package transformations match!")


## 3. Reusable Feature Engineering Module

### Objective

Move the validated notebook feature transformations into a reusable Python package module.

### Implementation

- Module: `src/flight_delay/features/build.py`
- Tests: `tests/test_feature_build.py`

### Feature Schema

The baseline transformation generates 17 features:

- 3 categorical features
- 2 original numerical features
- 4 calendar features
- 8 cyclical features

### Engineering Principles

**Consistent Schema**

The output feature columns have a fixed order across training, validation, and inference.

**Leakage Prevention**

Only the approved baseline input columns are accepted. Actual flight outcomes and target labels are rejected.

**Stateless Transformation**

The feature function does not learn parameters from the input dataset.

**Input Validation**

Invalid scheduled times, missing required fields, and missing feature values raise explicit errors.

**Reproducibility**

The production implementation is compared against the original notebook implementation to verify equivalent results.

### Limitations

The scheduled flight fields are derived from retrospective BTS records and are not verified historical T-24h schedule snapshots.

Categorical encoding and numerical scaling will be implemented in a separate model preprocessing pipeline, fitted using training data only.



## 4. Temporal Dataset Construction

### Objective

Construct chronological machine learning datasets using the approved 2023–2025 BTS flight records.

### Data Split Policy

| Split | Period | Purpose |
|---|---|---|
| Training | 2023-01 to 2023-12 | Model fitting |
| Validation | 2024-01 to 2024-08 | Model selection and tuning |
| Calibration | 2024-09 to 2024-10 | Probability calibration |
| Threshold | 2024-11 to 2024-12 | Classification threshold selection |
| Test | 2025-01 to 2025-12 | Final out-of-time evaluation |

### Implementation

`src/flight_delay/data/dataset.py`

The loader:

1. Resolves the months for a requested split.
2. Loads the corresponding processed Parquet files.
3. Validates flight dates and cohort eligibility.
4. Applies the reusable baseline feature transformation.
5. Returns aligned feature matrices, labels, and flight-date metadata.

### Leakage Prevention

- No random splitting is used.
- The feature builder accepts only approved scheduled-flight fields.
- Targets are stored separately from model features.
- The 2025 test set is locked by default.
- Preprocessing parameters for future encoders and scalers must be fitted on training data only.

### Limitations

Chronological splitting alone does not guarantee point-in-time correctness.

Any historical operational features introduced later must respect the T-24h prediction timestamp and source data availability.

Retrospective BTS scheduled-flight fields are treated as historical schedule proxies rather than verified T-24h snapshots.


In [ ]:
from flight_delay.data.dataset import (
    load_temporal_split,
    months_for_split,
)
from flight_delay.features.build import FEATURE_COLUMNS

# Verify the expected training months.
print("Training months:", months_for_split("train"))

train = load_temporal_split(
    "train",
    processed_dir=ROOT / "data" / "processed",
)

print("\n=== Training Dataset ===")
print("Feature shape:", train.X.shape)
print("Departure target:", train.y_departure.shape)
print("Arrival target:", train.y_arrival.shape)

print("Earliest flight:", train.flight_dates.min())
print("Latest flight:", train.flight_dates.max())

# Validate the full-year dataset.
assert train.X.shape == (1_478_412, 17)
assert len(train.y_departure) == len(train.X)
assert len(train.y_arrival) == len(train.X)

assert tuple(train.X.columns) == FEATURE_COLUMNS
assert train.X.isna().sum().sum() == 0

assert train.flight_dates.dt.year.eq(2023).all()

# Ensure targets remain separate.
assert "DepDel15" not in train.X.columns
assert "ArrDel15" not in train.X.columns

print("\n2023 training dataset validation passed!")


## 5. Train-Only Model Preprocessing

### Objective

Prepare the 17 engineered features for machine learning using categorical encoding and numerical scaling.

### Implementation

`src/flight_delay/models/preprocessing.py`

### Preprocessing Strategy

**Categorical Features**
- One-Hot Encoding
- Unknown categories ignored during transformation
- Sparse output to reduce memory consumption

**Numerical Features**
- StandardScaler
- Mean and standard deviation learned exclusively from training data

### Leakage Prevention

The preprocessor is fitted on the 2023 training split only.

Validation, calibration, threshold-selection, and final test datasets must use the fitted transformer without refitting.

### Engineering Design

The preprocessing transformer will later be combined with each machine learning estimator in a Scikit-learn Pipeline.

This ensures that model inference uses the same preprocessing parameters as training.


In [ ]:
from scipy import sparse

from flight_delay.models.preprocessing import (
    make_model_preprocessor,
    validate_model_features,
)

validate_model_features(train.X)

preprocessor = make_model_preprocessor()

# Fit using 2023 training data only.
preprocessor.fit(train.X)

# Transform 5,000 records for verification.
X_train_pilot = preprocessor.transform(
    train.X.iloc[:5000]
)

feature_names = preprocessor.get_feature_names_out()

print("=== Training Preprocessing ===")
print("Original features:", train.X.shape[1])
print("Encoded features:", len(feature_names))
print("Pilot shape:", X_train_pilot.shape)
print("Sparse matrix:", sparse.issparse(X_train_pilot))

assert X_train_pilot.shape[0] == 5000
assert X_train_pilot.shape[1] == len(feature_names)
assert sparse.issparse(X_train_pilot)

print("\nTraining preprocessing passed!")

In [ ]:
import numpy as np
from scipy import sparse

from flight_delay.data.storage import load_parquet
from flight_delay.data.dataset import BASELINE_INPUT_COLUMNS
from flight_delay.features.build import build_baseline_features
from flight_delay.models.preprocessing import validate_model_features

# Load January 2024 validation data.
validation_raw = load_parquet(
    ROOT / "data/processed/bts_2024_01_completed.parquet"
)

# Select approved input columns only.
X_validation_raw = validation_raw.loc[
    :,
    list(BASELINE_INPUT_COLUMNS),
].iloc[:5000].copy()

# Apply the same feature engineering logic.
X_validation_pilot = build_baseline_features(
    X_validation_raw
)

validate_model_features(X_validation_pilot)

# Capture training statistics before transformation.
scaler = preprocessor.named_transformers_["numeric"]

training_mean_before = scaler.mean_.copy()
training_scale_before = scaler.scale_.copy()

# IMPORTANT: Transform only, do not fit again.
X_validation_encoded = preprocessor.transform(
    X_validation_pilot
)

# Verify same encoded feature structure.
assert X_validation_encoded.shape == (5000, 312)
assert sparse.issparse(X_validation_encoded)

# Verify training statistics were unchanged.
np.testing.assert_array_equal(
    scaler.mean_,
    training_mean_before,
)

np.testing.assert_array_equal(
    scaler.scale_,
    training_scale_before,
)

print("=== Validation Preprocessing ===")
print("Raw features:", X_validation_pilot.shape[1])
print("Encoded features:", X_validation_encoded.shape[1])
print("Validation pilot shape:", X_validation_encoded.shape)
print("Sparse matrix:", sparse.issparse(X_validation_encoded))
print("Training statistics unchanged: True")

print("\nValidation preprocessing passed!")


### Validation Preprocessing Verification

The preprocessing transformer was fitted exclusively on the 2023 training dataset.

The fitted transformer generated 312 encoded features, including 298 categorical encoding columns and 14 numerical features.

To verify consistency, a sample of 5,000 flights from January 2024 was transformed without refitting.

The verification checks:

- Identical encoded feature dimensionality
- Sparse matrix output
- Unchanged training means and scaling parameters
- Consistent feature engineering and preprocessing rules

This confirms that the fitted preprocessing configuration can be reused on later data without learning additional statistics from the validation set.
